In [0]:
%run ../../utils/config.py

In [0]:
# Bygger dimensionstabeller och faktatabell enligt dimensional modellen
# Skapar även views för km-events och h-events

from pyspark.sql import functions as F

# Läs från silver
df = spark.read.table(SILVER_TABLE)

# dim_event
dim_event = df.select(
    "event_id",
    "event_name",
    "event_distance_length",
    "event_unit",
    "event_num_finishers",
    "year_of_event",
    "event_dates",
).distinct()

dim_event.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMAS['gold']}.dim_event")

print(f"dim_event: {dim_event.count():,} rader")

# dim_athlete
dim_athlete = df.select(
    "athlete_id",
    "athlete_country",
    "athlete_gender",
    "athlete_year_of_birth",
    "athlete_approx_age",
    "athlete_age_category",
).distinct()

dim_athlete.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMAS['gold']}.dim_athlete")

print(f"dim_athlete: {dim_athlete.count():,} rader")

# fct_results
fct_results = df.select(
    "result_id",
    "event_id",
    "athlete_id",
    "performance_seconds",
    "performance_km",
    "athlete_avg_speed",
)

fct_results.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMAS['gold']}.fct_results")

print(f"fct_results: {fct_results.count():,} rader")

# Views: en per event typ
# View för distans events (km och mi) – här är performance i sekunder
spark.sql(f"""
    CREATE OR REPLACE VIEW {SCHEMAS['gold']}.vw_distance_events AS
    SELECT
        f.result_id,
        f.performance_seconds,
        ROUND(f.performance_seconds / 3600.0, 2) AS performance_hours,
        f.athlete_avg_speed,
        e.event_name,
        e.event_distance_length,
        e.event_unit,
        e.year_of_event,
        a.athlete_country,
        a.athlete_gender,
        a.athlete_approx_age,
        a.athlete_age_category
    FROM {SCHEMAS['gold']}.fct_results f
    JOIN {SCHEMAS['gold']}.dim_event   e ON f.event_id   = e.event_id
    JOIN {SCHEMAS['gold']}.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE e.event_unit IN ('km', 'mi')
""")
print("View skapad: vw_distance_events")

# View för tids-events och här är performance i km
spark.sql(f"""
    CREATE OR REPLACE VIEW {SCHEMAS['gold']}.vw_timed_events AS
    SELECT
        f.result_id,
        f.performance_km,
        f.athlete_avg_speed,
        e.event_name,
        e.event_distance_length,
        e.event_unit,
        e.year_of_event,
        a.athlete_country,
        a.athlete_gender,
        a.athlete_approx_age,
        a.athlete_age_category
    FROM {SCHEMAS['gold']}.fct_results f
    JOIN {SCHEMAS['gold']}.dim_event   e ON f.event_id   = e.event_id
    JOIN {SCHEMAS['gold']}.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE e.event_unit = 'h'
""")
print("View skapad: vw_timed_events")

# Snabb kontroll
spark.sql(f"SELECT * FROM {SCHEMAS['gold']}.vw_distance_events LIMIT 5").display()
spark.sql(f"SELECT * FROM {SCHEMAS['gold']}.vw_timed_events LIMIT 5").display()